# 01 - Process the demo PDFs

PDFs -> Content Understanding Markdown -> sections and titles -> chunks -> embeddings.

Based on the [Traditional RAG processing example](https://github.com/jomedinagomez/RAG_process/tree/main/Traditional_RAG). Run the cells after configuring `.env`; extraction and embeddings make real Azure calls. This notebook does not create or populate a search index.

In [ ]:
import json
import os
import re
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import AzureCliCredential, get_bearer_token_provider
from azure.ai.contentunderstanding import ContentUnderstandingClient
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from openai import AzureOpenAI
from IPython.display import Markdown, display

ROOT = Path.cwd() if (Path.cwd() / 'samples').is_dir() else Path.cwd().parent
OUTPUT = ROOT / 'output'
OUTPUT.mkdir(exist_ok=True)
load_dotenv(ROOT / '.env')
credential = AzureCliCredential(subscription=os.getenv('AZURE_SUBSCRIPTION_ID') or None, process_timeout=60)
embedding_deployment = os.environ['EMBEDDING_DEPLOYMENT']
extraction_deployment = os.environ['EXTRACTION_DEPLOYMENT']
dimensions = 3072

documents = json.loads((ROOT / 'samples' / 'metadata.json').read_text(encoding='utf-8'))
len(documents)

## Extract Markdown

Use `documents = documents[:3]` here if you want a small first run. Each real analysis result and Markdown file is saved under `output/extractions`.

In [ ]:
cu_client = ContentUnderstandingClient(
    endpoint=os.environ['CONTENTUNDERSTANDING_ENDPOINT'],
    credential=credential,
    api_version='2025-11-01',
)
# Prebuilt analyzers ask for models by alias; point every alias at our deployments.
cu_client.update_defaults(model_deployments={
    'gpt-5.5': extraction_deployment,
    'prebuilt-analyzer-completion': extraction_deployment,
    'prebuilt-analyzer-completion-mini': extraction_deployment,
    'text-embedding-3-large': embedding_deployment,
    'prebuilt-analyzer-embedding': embedding_deployment,
})

extraction_folder = OUTPUT / 'extractions'
extraction_folder.mkdir(exist_ok=True)
extracted_documents = []
for document in documents:
    pdf = ROOT / 'samples' / document['file']
    poller = cu_client.begin_analyze_binary(
        analyzer_id='prebuilt-documentSearch', binary_input=pdf.read_bytes(),
        content_type='application/pdf',
    )
    result = poller.result(timeout=600)
    markdown = result.contents[0].markdown or ''
    (extraction_folder / f'{document["document_id"]}.md').write_text(markdown, encoding='utf-8')
    (extraction_folder / f'{document["document_id"]}.json').write_text(
        json.dumps(result.as_dict(), indent=2, ensure_ascii=False), encoding='utf-8'
    )
    extracted_documents.append({'document': document, 'markdown': markdown})
    print(document['title'])
cu_client.close()

In [ ]:
display(Markdown(extracted_documents[0]['markdown'][:6000]))


## Split into sections and chunks

Keep the document title and Markdown heading with each chunk, and copy the document's citation URL and access fields onto every chunk.

In [ ]:
header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[('#', 'Title'), ('##', 'Section'), ('###', 'Subsection')]
)
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base', chunk_size=800, chunk_overlap=80
)

chunks = []
for item in extracted_documents:
    document = item['document']
    markdown = re.sub(r'(?m)^([^\n]+)\n={3,}\s*$', r'# \1', item['markdown'])
    ordinal = 0
    for section in header_splitter.split_text(markdown):
        header = ' > '.join(section.metadata.values()) or 'Overview'
        for text in text_splitter.split_text(section.page_content):
            chunks.append({
                'chunk_id': f'{document["document_id"]}-{ordinal:04d}',
                'document_id': document['document_id'], 'title': document['title'],
                'section_title': header, 'content': document['title'] + '\n' + header + '\n\n' + text,
                'url': document['url'], 'metadata_storage_path': document['url'],
                'group_ids': document['group_ids'], 'persona': document['persona'],
                'content_source': document['content_source'], 'policy_family': document['policy_family'],
                'revision': document['revision'], 'parent_document_id': document.get('parent_document_id'),
            })
            ordinal += 1

(OUTPUT / 'chunks.json').write_text(json.dumps(chunks, indent=2, ensure_ascii=False), encoding='utf-8')
print('Chunks prepared:', len(chunks))
chunks[:1]

## Generate embeddings and save the upload file

`text-embedding-3-large`, 3072 dimensions. Next run **02_create_index**, then **03_populate_and_test_index**.

In [ ]:
token_provider = get_bearer_token_provider(credential, 'https://cognitiveservices.azure.com/.default')
model_client = AzureOpenAI(
    azure_endpoint=os.environ['AZURE_OPENAI_ENDPOINT'],
    api_version=os.getenv('AZURE_OPENAI_API_VERSION', '2024-10-21'),
    azure_ad_token_provider=token_provider,
)
for offset in range(0, len(chunks), 16):
    batch = chunks[offset:offset + 16]
    response = model_client.embeddings.create(
        model=embedding_deployment, input=[chunk['content'] for chunk in batch], dimensions=dimensions
    )
    for embedding in response.data:
        batch[embedding.index]['content_vector'] = embedding.embedding

model_client.close()
(OUTPUT / 'embedded_chunks.json').write_text(json.dumps(chunks, ensure_ascii=False), encoding='utf-8')
print('Saved output/embedded_chunks.json:', len(chunks), 'chunks')